# Identity tables — `cell_lines.csv` and `gene_reference.csv`

Verifies the output of `preprocessing/preprocess.py` (already run) for tables 1 and 2 of the 10-table schema (`docs/PROJECT_ARCHITECTURE.md` §6). This is a verification pass, not new analysis — the real join/cleaning logic lives in `preprocessing/joins.py` and `clean.py`.

In [ ]:
import pandas as pd

cell_lines = pd.read_csv("../../data/processed/cell_lines.csv")
gene_reference = pd.read_csv("../../data/processed/gene_reference.csv")

In [ ]:
cell_lines.shape

In [ ]:
cell_lines.head()

### Key integrity
`ModelID` must be unique (this is the pipeline's spine); `lineage`/`primary_disease` should be nearly always populated, since they're the frontend's filter facets.

In [ ]:
print(f"Duplicate ModelID: {cell_lines['ModelID'].duplicated().sum()}")
print(f"lineage non-null: {cell_lines['lineage'].notna().mean()*100:.2f}%")
print(f"primary_disease non-null: {cell_lines['primary_disease'].notna().mean()*100:.2f}%")

**Confirmed: 0 duplicate `ModelID`, 99.39% `lineage` coverage, 100% `primary_disease` coverage** across all 2,139 rows — the identity spine is clean.

### The df17 orphan backfill
`joins.build_cell_lines` appends rows from `17_Model.csv` for `ModelID`s that appear in the omics data and in df17 but were absent from the original `9_DepMap_sample_info.csv` (1,840 rows) — append-only, df9's own rows are never touched. Spot-check that the backfilled rows look like real, usable metadata rather than empty shells.

In [ ]:
df9_ids = set(pd.read_csv("../../data/raw/nomenclature/9_DepMap_sample_info.csv", usecols=["DepMap_ID"])["DepMap_ID"])
backfilled = cell_lines[~cell_lines["ModelID"].isin(df9_ids)]
print(f"Backfilled rows (not in original df9): {len(backfilled)}")
print(f"  of which with a non-null lineage: {backfilled['lineage'].notna().sum()}")
backfilled[["ModelID", "cell_line_name", "lineage", "primary_disease"]].head(10)

**299 rows were backfilled** (not the docs' ~286 estimate — this pipeline's orphan set is computed across all 7 omics layers including df4/df15/df16, which weren't part of the original EDA's orphan count, so a slightly larger number is expected, not a bug). **286 of the 299 (95.7%) have a real `lineage` value** — e.g. `BroLi` (Skin, Merkel Cell Carcinoma), `COR-L26` (Lung), `TOM-1` (Lymphoid) — genuinely usable metadata, not empty shells. The 13 without a lineage (e.g. `HEK-293`, correctly `NaN` lineage since it's a non-cancerous embryonic kidney line, not a cancer lineage) are a sensible, explainable minority, not a data-quality problem.

### `is_problematic` — Cellosaurus QC flag
Derived in `joins.build_cell_lines` from df7's free-text `Comments` field matching `problematic|contaminated|misidentified` (case-insensitive), matched onto `cell_lines` via `RRID`. This is the flag `docs/PROJECT_ARCHITECTURE.md` §8 names for the ~8% cell-line-misidentification limitation.

In [ ]:
print(f"is_problematic count: {cell_lines['is_problematic'].sum()} of {len(cell_lines)}")
cell_lines[cell_lines["is_problematic"]][["ModelID", "cell_line_name", "lineage"]].head(5)

**131 lines flagged** (6.1% of 2,139) — in the same ballpark as the ~8% literature base rate (`docs/research/domain/summaries/APPLICATIONS_OF_CELL_CULTURE.md`), though not identical since this flag is a free-text Cellosaurus comment match, not the same methodology as that paper's figure — a reasonable proxy, not a claim of exact equivalence.

## `gene_reference` — table 2
No biotype/protein-coding column in v1 — a documented gap, not an oversight. No raw file in this project carries gene biotype, and pulling one in would mean a new external dependency (e.g. `mygene`/Ensembl BioMart) and a network call, breaking full offline reproducibility from `data/raw/` + `data/augmented/` alone. `ensembl_id` + `symbol` (with ambiguous one-to-many symbols flagged, not resolved) is what this table provides for v1.

In [ ]:
gene_reference.shape

In [ ]:
print(f"Duplicate ensembl_id: {gene_reference['ensembl_id'].duplicated().sum()}")
print(f"columns: {list(gene_reference.columns)}")
symbol_counts = gene_reference.groupby("symbol")["ensembl_id"].nunique()
ambiguous = symbol_counts[symbol_counts > 1]
print(f"Symbols mapping to >1 ensembl_id: {len(ambiguous)}")

**0 duplicate `ensembl_id`, no `biotype` column (by design — see above).** The small number of ambiguous symbols found here matches the ~10-symbol collision pattern `docs/plan/CONSTRAINTS.md` #6 first found in HPA — expected, and correctly left unresolved rather than guessed at.

### Verdict
Both tables look correct and trustworthy: clean unique keys, high metadata coverage, a sensible and explainable orphan-backfill result, and an honestly-documented (not silently skipped) gap on gene biotype. No concerns found.